In [39]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer


In [40]:
df=pd.read_csv("./CSVs/Pokemon.csv",usecols=['Total','HP',"Attack","Defense","Sp. Atk","Sp. Def","Speed", "Generation","Legendary"])

In [41]:
df.sample(9)

,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
633,300,55,50,40,40,40,75,5,False
332,530,70,110,180,60,60,50,3,False
590,445,103,60,86,60,86,50,5,False
431,600,50,95,90,95,90,180,3,True
160,420,61,84,65,70,70,70,1,False
345,302,70,43,53,43,53,40,3,False
239,450,100,100,80,60,60,50,2,False
178,442,100,50,50,76,96,70,2,False
524,530,110,130,80,70,60,80,4,False


In [42]:
# df=df.drop(columns=["HP","Attack","Defense","Sp. Atk","Sp. Def"])

In [43]:
df.head()

,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,318,45,49,49,65,65,45,1,False
1,405,60,62,63,80,80,60,1,False
2,525,80,82,83,100,100,80,1,False
3,625,80,100,123,122,120,80,1,False
4,309,39,52,43,60,50,65,1,False


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Total       800 non-null    int64
 1   HP          800 non-null    int64
 2   Attack      800 non-null    int64
 3   Defense     800 non-null    int64
 4   Sp. Atk     800 non-null    int64
 5   Sp. Def     800 non-null    int64
 6   Speed       800 non-null    int64
 7   Generation  800 non-null    int64
 8   Legendary   800 non-null    bool 
dtypes: bool(1), int64(8)
memory usage: 50.9 KB


In [45]:
X=df.drop(columns=["Legendary"])

In [46]:
Y=df['Legendary']

In [47]:
X.shape,Y.shape

((800, 8), (800,))

In [48]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.10,random_state=42)

In [49]:
X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((720, 8), (80, 8), (720,), (80,))

In [50]:
DTC=DecisionTreeClassifier()

In [51]:
DTC.fit(X_train,Y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [52]:
pred=DTC.predict(X_test)

In [53]:
accuracy_score(pred,Y_test)

0.9625

In [64]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,Y,cv=10,scoring="accuracy"))

np.float64(0.95)

In [65]:
des_for_HP=KBinsDiscretizer(n_bins=10,encode="ordinal",strategy="kmeans")
des_for_Total=KBinsDiscretizer(n_bins=10,encode="ordinal",strategy='kmeans')

In [68]:
trf=ColumnTransformer([
    ('first',des_for_Total,[0]),
    ('second',des_for_HP,[1]),
],remainder="passthrough")

In [70]:
X_train_trf=trf.fit_transform(X_train)

C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


In [82]:
X_train_trf.shape,X_train.shape

((720, 8), (720, 8))

In [71]:
X_test_trf=trf.transform(X_test)

In [76]:
df_new=X_train.iloc[:,[0,1]]

In [84]:
# new_df=pd.DataFrame({
#     "Total":X_train.Total,
#     "Total_train":X_train_trf[0],
#     "HP":X_train.HP,
#     # "HP_train":X_train_trf[1]
# })

In [87]:
X_tran_df=pd.DataFrame(X_train_trf,columns=X_train.columns)

In [88]:
X_tran_df

,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,6.0,2.0,150.0,50.0,150.0,50.0,150.0,3.0
1,6.0,3.0,155.0,120.0,65.0,90.0,105.0,1.0
2,2.0,3.0,80.0,50.0,35.0,35.0,35.0,1.0
3,5.0,4.0,105.0,65.0,100.0,70.0,70.0,1.0
4,5.0,3.0,120.0,65.0,45.0,85.0,125.0,4.0
...,...,...,...,...,...,...,...,...
715,6.0,2.0,50.0,65.0,175.0,95.0,150.0,1.0
716,2.0,1.0,105.0,90.0,25.0,25.0,50.0,1.0
717,8.0,5.0,130.0,90.0,110.0,154.0,90.0,2.0
718,2.0,2.0,58.0,44.0,58.0,44.0,61.0,4.0


In [96]:
pd.DataFrame(
    {"TR_T":trf.named_transformers_['first'].bin_edges_[0].tolist(),        
     "TR_H":trf.named_transformers_['second'].bin_edges_[0].tolist()})

,TR_T,TR_H
0,180.000000,1.000000
1,241.854839,18.598837
2,298.415650,39.736815
3,363.790169,56.146351
4,434.652781,72.817197
5,489.329464,91.314776
6,550.652519,112.203321
7,607.169556,137.447368
8,655.923077,171.000000
9,731.750000,221.250000


In [90]:
trf.named_transformers_['second'].bin_edges_

array([array([  1.        ,  18.59883721,  39.73681535,  56.14635135,
               72.81719674,  91.31477591, 112.2033208 , 137.44736842,
              171.        , 221.25      , 255.        ])             ],
      dtype=object)

In [97]:
DTC2=DecisionTreeClassifier()

In [98]:
DTC2.fit(X_tran_df,Y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [99]:
DTC.predict(X_test)

array([ True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
       False, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,  True, False, False, False, False,  True, False,
        True, False, False, False, False, False, False, False])

In [100]:
accuracy_score(Y_test,pred)

0.9625

In [103]:
np.mean(cross_val_score(DecisionTreeClassifier(),X_tran_df,Y_train,cv=10,scoring='accuracy'))

np.float64(0.9430555555555555)